## Trying other further ways to improve the Decision Tree and the Random Forest
## 1. Try of Improvement:
### Using other Dataset to train on:
we use the dataset from n4.4_leakage_audit.ipynp with some different columns to see if it has better information for prediction

In [1]:
import pandas as pd
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import GridSearchCV
from sklearn.metrics import classification_report


ml_df = pd.read_csv("../data/ml_df_v4.csv")

df_train = ml_df[ml_df["year"] <= 2018]
prediction = ml_df[ml_df['year'] == 2022]

y_train = df_train["result_target"]
y_test = prediction["result_target"]

#Using the features from n4.4_leakage_audit.ipynb 
LEAKAGE_COLS = {
    "home_goals", "away_goals", "score", "result",
    "home_team_win", "away_team_win", "draw",
    "extra_time", "penalty_shootout", "score_penalties",
}

CATEGORICAL_COLS = ["stage", "tournament_name", "host_country", "home_team", "away_team", "tournament_size_category"]
ENGINEERED_PREFIXES = ["home_", "away_"]
EXPLICIT_COLS = [
    "win_rate_diff", "goal_diff_diff", "form_diff", "goals_per_match_diff", "conceded_per_match_diff",
    "season_win_rate_diff", "season_goal_diff_diff", "elo_diff", "ranking_diff", "wc_goals_diff_before",
    "total_teams", "matches_played", "goals_scored_tournament", "avg_goals_per_game", "year_normalized",
    "is_neutral_venue", "home_advantage_strength",
]

feature_cols = [
    c for c in ml_df.columns
    if c in CATEGORICAL_COLS or any(c.startswith(p) for p in ENGINEERED_PREFIXES) or c in EXPLICIT_COLS
]

feature_cols = [
    c for c in feature_cols 
    if c not in LEAKAGE_COLS 
    and c != "result_target"
    and not c.endswith("_x")  
    and not c.endswith("_y")
    and "h2h" not in c        
    and "season" not in c     
]

x_train = df_train[feature_cols].select_dtypes(exclude=['object'])
x_test = prediction[feature_cols].select_dtypes(exclude=['object'])


In [2]:

# Train : Training the model on data until 2018
df_train = ml_df[ml_df["year"] <= 2018]

# Prediction : Predicting the results of World Cup 2022
prediction = ml_df[ml_df['year'] == 2022]

y_train = df_train["result_target"]
y_test = prediction["result_target"]


#The best parameters for the Random Forest we could get from before (from v1)
optimized_forest = RandomForestClassifier(
    n_estimators=300,            # 300 Trees to maximize accuracy
    max_depth=3,                 # depth of 3 was the best for a single tree, so we use it for all trees in the forest now
    max_features="sqrt",         # forcing the model to be creative by limitting the number of features it can learn from at each split,
    min_samples_leaf= 3,         
    class_weight= "balanced",    # We saw at the decision tree that balancing the class-weight is the best for predicting the results
    random_state=42,
    n_jobs=-1                    
)

optimized_forest.fit(x_train, y_train)

y_pred_optimized_forest = optimized_forest.predict(x_test)

print("Classification Report for the optimized random forest model: \n" , classification_report(y_test, y_pred_optimized_forest))

#Data Leakage?
importances = pd.Series(optimized_forest.feature_importances_, index=x_train.columns)
print("Most relevant columns for the model, helps checking if there is any data leakage")
print(importances.sort_values(ascending=False).head(10))

Classification Report for the optimized random forest model: 
               precision    recall  f1-score   support

     AwayWin       0.44      0.70      0.54        20
        Draw       0.31      0.33      0.32        15
     HomeWin       0.81      0.45      0.58        29

    accuracy                           0.50        64
   macro avg       0.52      0.49      0.48        64
weighted avg       0.58      0.50      0.51        64

Most relevant columns for the model, helps checking if there is any data leakage
year_normalized                    0.073536
matches_played                     0.068881
away_total_win_rate_before         0.066257
away_goal_diff_per_match_before    0.057220
total_teams                        0.056131
away_goals_per_match_before        0.051023
away_goal_diff_before              0.046586
goals_scored_tournament            0.040336
away_total_draws_before            0.038386
avg_goals_per_game                 0.035479
dtype: float64


## Using another dataset did not help our model to predict better instead it got worse. 

## So our final try of improvement for the model is, to use less columns than before.
We will try to give the model less columns to train with and see if it gives us better result.

Our best Random Forest until now is the one below with 59% accuracy:

In [3]:
df = pd.read_csv("ml_df_test_bab.csv")

# Train : Training the model on data until 2018
df_train = df[df["year"] <= 2018]

# Prediction : Predicting the results of World Cup 2022
prediction = df[df['year'] == 2022]

y_train = df_train["result_target"]
y_test = prediction["result_target"]

# Feature Selection
relevant_columns = [
    'elo_diff', 'win_rate_diff', 'goal_diff_diff', 'form_diff', 
    'goals_per_match_diff', 'conceded_per_match_diff',
    'home_elo_before', 'away_elo_before',
    'home_total_win_rate_before', 'away_total_win_rate_before',
    'home_goal_diff_per_match_before', 'away_goal_diff_per_match_before',
    'home_last5_win_rate', 'away_last5_win_rate',
    'home_last5_goal_diff', 'away_last5_goal_diff',
    'home_tournaments_played_before', 'away_tournaments_played_before',
    'home_has_won_world_cup_before', 'away_has_won_world_cup_before',
    'home_is_defending_champion', 'away_is_defending_champion'
]

#Chosing the relevant columns for training ( excluding strings/ spoilers)
x_train = df_train[relevant_columns].select_dtypes(exclude=['object'])
x_test = prediction[relevant_columns].select_dtypes(exclude=['object'])

optimized_forest = RandomForestClassifier(
    n_estimators=300,            
    max_depth=3,                 
    max_features="sqrt",        
    min_samples_leaf= 3,         
    class_weight= "balanced",    
    random_state=42,
    n_jobs=-1                    
)

optimized_forest.fit(x_train, y_train)

y_pred_optimized_forest = optimized_forest.predict(x_test)

print("Classification Report for the optimized random forest model: \n" , classification_report(y_test, y_pred_optimized_forest))


Classification Report for the optimized random forest model: 
               precision    recall  f1-score   support

     AwayWin       0.50      0.70      0.58        20
        Draw       0.45      0.33      0.38        15
     HomeWin       0.76      0.66      0.70        29

    accuracy                           0.59        64
   macro avg       0.57      0.56      0.56        64
weighted avg       0.61      0.59      0.59        64



# 2. Improvement
## Removing all columns which contain information that the model already has,
for example: home_elo_before and away_elo_before -> give the same information as : elo_dif

--> So we are removing:
- home_elo_before & away_elo_before

- home_total_win_rate_before & away_total_win_rate_before

- home_goal_diff_per_match_before & away_goal_diff_per_match_before

In [4]:
relevant_columns = [
    'elo_diff', 'win_rate_diff', 'goal_diff_diff', 'form_diff', 
    'goals_per_match_diff', 'conceded_per_match_diff',
    'home_last5_win_rate', 'away_last5_win_rate',
    'home_last5_goal_diff', 'away_last5_goal_diff',
    'home_tournaments_played_before', 'away_tournaments_played_before',
    'home_has_won_world_cup_before', 'away_has_won_world_cup_before',
    'home_is_defending_champion', 'away_is_defending_champion'
]

#Chosing the relevant columns for training ( excluding strings/ spoilers)
x_train = df_train[relevant_columns].select_dtypes(exclude=['object'])
x_test = prediction[relevant_columns].select_dtypes(exclude=['object'])

optimized_forest = RandomForestClassifier(
    n_estimators=300,            # 300 Trees to maximize accuracy
    max_depth=3,                 # depth of 3 was the best for a single tree, so we use it for all trees in the forest now
    max_features="sqrt",         # forcing the model to be creative by limitting the number of features it can learn from at each split,
    min_samples_leaf= 3,         # 
    class_weight= "balanced",     # We saw at the decision tree that balancing the class-weight is the best for predicting the results
    random_state=42,
    n_jobs=-1                    
)

optimized_forest.fit(x_train, y_train)

y_pred_optimized_forest = optimized_forest.predict(x_test)

print("Classification Report for the optimized random forest model: \n" , classification_report(y_test, y_pred_optimized_forest))

Classification Report for the optimized random forest model: 
               precision    recall  f1-score   support

     AwayWin       0.48      0.60      0.53        20
        Draw       0.35      0.40      0.38        15
     HomeWin       0.68      0.52      0.59        29

    accuracy                           0.52        64
   macro avg       0.50      0.51      0.50        64
weighted avg       0.54      0.52      0.52        64



## The accuracy got worse.
Apparently not only the Elo-Dif matters, but also how high the elo is. 
It seems like the model needed the information how high the elo is in the matches.
For example:
1. Scenario:
Away-Elo: 1500, Home-Elo: 1600 -> Elo-Dif : 100
2. Scenario:
Away-Elo : 2500, Home-Elo: 2600 -> Elo-Dif : 100, 

We have in both cases the information of Elo-Dif = 100, but the model doesnt know how high the Elo is, 
is the Elo > 1000? or Elo > 2000 ?, the model doesnt know, but it needs this information for an accurate prediction.

## 2. Improvement also did not work

# 3. Improvement: 
Removing columns that rarely contain information:
Columns like : 
    - 'home_has_won_world_cup_before', 'away_has_won_world_cup_before',
    - 'home_is_defending_champion', 'away_is_defending_champion'

There is in every World-Cup only one defending champion, so the columns equal most of the times 0.
There are also not that many countries that won the world-cup, so the columns equal most of the time again 0.

In [5]:
relevant_columns = [
    'elo_diff', 'win_rate_diff', 'goal_diff_diff', 'form_diff', 
    'goals_per_match_diff', 'conceded_per_match_diff',
    'home_elo_before', 'away_elo_before',
    'home_total_win_rate_before', 'away_total_win_rate_before',
    'home_goal_diff_per_match_before', 'away_goal_diff_per_match_before',
    'home_last5_win_rate', 'away_last5_win_rate',
    'home_last5_goal_diff', 'away_last5_goal_diff',
    'home_tournaments_played_before', 'away_tournaments_played_before',
]

#Chosing the relevant columns for training ( excluding strings/ spoilers)
x_train = df_train[relevant_columns].select_dtypes(exclude=['object'])
x_test = prediction[relevant_columns].select_dtypes(exclude=['object'])

optimized_forest = RandomForestClassifier(
    n_estimators=300,            # 300 Trees to maximize accuracy
    max_depth=3,                 # depth of 3 was the best for a single tree, so we use it for all trees in the forest now
    max_features="sqrt",         # forcing the model to be creative by limitting the number of features it can learn from at each split,
    min_samples_leaf= 3,         # 
    class_weight= "balanced",     # We saw at the decision tree that balancing the class-weight is the best for predicting the results
    random_state=42,
    n_jobs=-1                    
)

optimized_forest.fit(x_train, y_train)

y_pred_optimized_forest = optimized_forest.predict(x_test)

print("Classification Report for the optimized random forest model: \n" , classification_report(y_test, y_pred_optimized_forest))

Classification Report for the optimized random forest model: 
               precision    recall  f1-score   support

     AwayWin       0.50      0.70      0.58        20
        Draw       0.36      0.27      0.31        15
     HomeWin       0.72      0.62      0.67        29

    accuracy                           0.56        64
   macro avg       0.53      0.53      0.52        64
weighted avg       0.57      0.56      0.56        64



## The 3. improvement also didnt work:
Even though the removed columns are most of the time empty, they still are an important part of the prediction. 
So when removed, the accuracy gets worse.

# Overall we couldnt find any ways of improving the Random_Forest we got before.
## So the best accuracy we could have get until now is 59% with the following Random_Forest from before: 

In [6]:
df = pd.read_csv("ml_df_test_bab.csv")

relevant_columns = [
    'elo_diff', 'win_rate_diff', 'goal_diff_diff', 'form_diff', 
    'goals_per_match_diff', 'conceded_per_match_diff',
    'home_elo_before', 'away_elo_before',
    'home_total_win_rate_before', 'away_total_win_rate_before',
    'home_goal_diff_per_match_before', 'away_goal_diff_per_match_before',
    'home_last5_win_rate', 'away_last5_win_rate',
    'home_last5_goal_diff', 'away_last5_goal_diff',
    'home_tournaments_played_before', 'away_tournaments_played_before',
    'home_has_won_world_cup_before', 'away_has_won_world_cup_before',
    'home_is_defending_champion', 'away_is_defending_champion'
]

x_train = df_train[relevant_columns].select_dtypes(exclude=['object'])
x_test = prediction[relevant_columns].select_dtypes(exclude=['object'])

optimized_forest = RandomForestClassifier(
    n_estimators=300,            
    max_depth=3,                 
    max_features="sqrt",        
    min_samples_leaf= 3,         
    class_weight= "balanced",     
    random_state=42,
    n_jobs=-1                    
)

optimized_forest.fit(x_train, y_train)

y_pred_optimized_forest = optimized_forest.predict(x_test)

print("Classification Report for the optimized random forest model: \n" , classification_report(y_test, y_pred_optimized_forest))


Classification Report for the optimized random forest model: 
               precision    recall  f1-score   support

     AwayWin       0.50      0.70      0.58        20
        Draw       0.45      0.33      0.38        15
     HomeWin       0.76      0.66      0.70        29

    accuracy                           0.59        64
   macro avg       0.57      0.56      0.56        64
weighted avg       0.61      0.59      0.59        64

